# 🚦 Traffic Sign Recognition System: Complete 10-Phase Pipeline
### B.Tech Project: Deep Learning & Convolutional Neural Networks (CNN)

Welcome! This notebook implements the **complete 10-phase project roadmap** for recognizing traffic signs using TensorFlow and Keras.

---

### 📋 Overview of the 10 Phases:
1. **Phase 1: Environment Setup & Library Imports**
2. **Phase 2: Dataset Acquisition** (Automated GTSRB download & extraction)
3. **Phase 3: Class Label Mapping** (43 traffic sign categories)
4. **Phase 4: Exploratory Data Analysis (EDA)** (Sample visualization & counts)
5. **Phase 5: Data Preprocessing Pipeline** (Resizing $32\times32$, Normalization, & Train/Test Split)
6. **Phase 6: CNN Architecture Design** (Conv2D, MaxPool, Dropout, Dense)
7. **Phase 7: Model Training & Checkpointing** (Training with Adam optimizer)
8. **Phase 8: Model Evaluation & Performance Metrics** (Accuracy, Loss Curves & Test Score)
9. **Phase 9: Single-Image Inference Engine** (Top-3 predictions with confidence)
10. **Phase 10: Model Export for Web Deployment** (Save `.keras` model for Flask app)

## 📌 Phase 1: Environment Setup & Library Imports
In this phase, we load essential machine learning and image processing packages.

In [1]:
import os
import zipfile
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow Version:", tf.__version__)
print("NumPy Version:", np.__version__)
print("OpenCV Version:", cv2.__version__)
print("✅ Phase 1 Complete: All libraries imported successfully!")

ModuleNotFoundError: No module named 'cv2'

## 📌 Phase 2: Dataset Acquisition & Extraction
We download the benchmark **German Traffic Sign Recognition Benchmark (GTSRB)** directly from the Harvard CS50 repository (~200MB, pre-organized into 43 folders `0` through `42`).

In [ ]:
DATASET_DIR = "dataset"
ZIP_FILE = "gtsrb.zip"
DOWNLOAD_URL = "https://cdn.cs50.net/ai/2020/x/projects/5/gtsrb.zip"

DATA_PATH = os.path.join(DATASET_DIR, "gtsrb")

if not os.path.exists(DATA_PATH):
    os.makedirs(DATASET_DIR, exist_ok=True)
    print("Downloading GTSRB dataset (Harvard CS50 archive)...")
    urllib.request.urlretrieve(DOWNLOAD_URL, ZIP_FILE)
    print("Extracting files...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    if os.path.exists(ZIP_FILE):
        os.remove(ZIP_FILE)  # Clean up zip
    print(f"✅ Phase 2 Complete: Dataset extracted at: {DATA_PATH}")
else:
    print(f"✅ Phase 2 Complete: Dataset already available at: {DATA_PATH}")

## 📌 Phase 3: Class Label Mapping & Metadata
Mapping the numerical IDs (`0` to `42`) to standard, human-readable traffic sign names.

In [ ]:
CLASS_NAMES = {
    0: 'Speed limit (20km/h)',
    1: 'Speed limit (30km/h)',
    2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)',
    5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',
    7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',
    9: 'No passing',
    10: 'No passing for vehicles over 3.5 tons',
    11: 'Right-of-way at intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Vehicles > 3.5 tons prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve left',
    20: 'Dangerous curve right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed & passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End no passing veh > 3.5 tons'
}

print(f"✅ Phase 3 Complete: Defined {len(CLASS_NAMES)} distinct traffic sign classes.")

## 📌 Phase 4: Exploratory Data Analysis (EDA)
We inspect image counts across classes and visualize sample traffic signs from the dataset.

In [ ]:
class_counts = {}
sample_images = {}

for class_id in range(43):
    folder = os.path.join(DATA_PATH, str(class_id))
    if os.path.exists(folder):
        files = os.listdir(folder)
        class_counts[class_id] = len(files)
        if len(files) > 0:
            sample_images[class_id] = os.path.join(folder, files[0])

total_images = sum(class_counts.values())
print(f"Total images found: {total_images}")
print(f"Min images in a class: {min(class_counts.values())} (Class {min(class_counts, key=class_counts.get)})")
print(f"Max images in a class: {max(class_counts.values())} (Class {max(class_counts, key=class_counts.get)})")

# Display 9 sample traffic signs
plt.figure(figsize=(10, 10))
demo_classes = [2, 14, 13, 1, 17, 25, 26, 33, 38]  # Common signs: 50km/h, Stop, Yield, etc.

for i, cid in enumerate(demo_classes):
    plt.subplot(3, 3, i + 1)
    img = cv2.imread(sample_images[cid])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.title(f"[{cid}] {CLASS_NAMES[cid]}", fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()
print("✅ Phase 4 Complete: EDA & sample inspection finished.")

## 📌 Phase 5: Image Preprocessing Pipeline
1. **Resize** every image to $32 \times 32$ pixels.
2. **Convert** BGR color space to RGB.
3. **Normalize** pixel values to $[0.0, 1.0]$.
4. **Split** into 80% Training and 20% Testing sets using stratified sampling.

In [ ]:
X_list = []
y_list = []
IMAGE_SIZE = (32, 32)

print("Processing images (resizing to 32x32 and converting color)...")

for class_id in range(43):
    folder = os.path.join(DATA_PATH, str(class_id))
    if not os.path.exists(folder):
        continue
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        if img is not None:
            img = cv2.resize(img, IMAGE_SIZE)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X_list.append(img)
            y_list.append(class_id)

# Convert to NumPy arrays and normalize
X = np.array(X_list, dtype='float32') / 255.0
y = np.array(y_list, dtype='int32')

# Split: 80% Train, 20% Test (stratified to maintain class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} images, shape: {X_train.shape}")
print(f"Testing set  : {X_test.shape[0]} images, shape: {X_test.shape}")
print("✅ Phase 5 Complete: Preprocessing pipeline finished.")

## 📌 Phase 6: CNN Architecture Design
We assemble a clear, effective Convolutional Neural Network:
- **Conv2D + MaxPool Block 1**: 32 filters ($3\times3$)
- **Conv2D + MaxPool Block 2**: 64 filters ($3\times3$)
- **Flatten & Dense**: 128 hidden neurons with ReLU
- **Dropout (0.3)**: Prevents overfitting
- **Dense(43, Softmax)**: Outputs probability distribution across 43 classes

In [ ]:
model = models.Sequential([
    # Input + Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Feature flattening
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output layer for 43 classes
    layers.Dense(43, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("✅ Phase 6 Complete: CNN compiled successfully.")

## 📌 Phase 7: Model Training & Checkpointing
We train for 15 epochs with a batch size of 32. We also use `EarlyStopping` to prevent overfitting if validation loss stops improving.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 32

# Stop early if validation loss does not improve for 4 consecutive epochs
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping]
)

print("✅ Phase 7 Complete: Model training finished.")

## 📌 Phase 8: Model Evaluation & Performance Metrics
We plot training vs. validation accuracy and loss curves, and evaluate the final test score.

In [ ]:
plt.figure(figsize=(12, 4))

# Accuracy Curve
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Testing Accuracy', linewidth=2)
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Curve
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Testing Loss', linewidth=2)
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Final evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Final Test Accuracy: {test_acc * 100:.2f}%")
print(f"📉 Final Test Loss:     {test_loss:.4f}")
print("✅ Phase 8 Complete: Evaluation and curves plotted.")

## 📌 Phase 9: Single-Image Inference Engine
A self-contained prediction function displaying the input sign, predicted class name, and confidence.

In [ ]:
def predict_traffic_sign(image_path):
    if not os.path.exists(image_path):
        print(f"Error: File not found -> {image_path}")
        return
    
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (32, 32))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_norm = img_rgb / 255.0
    input_batch = np.expand_dims(img_norm, axis=0)
    
    preds = model.predict(input_batch, verbose=0)[0]
    top_class = int(np.argmax(preds))
    confidence = float(preds[top_class] * 100)
    
    plt.figure(figsize=(3, 3))
    plt.imshow(img_rgb)
    plt.title(f"{CLASS_NAMES[top_class]}\n({confidence:.1f}% confidence)", color='green', fontsize=11)
    plt.axis('off')
    plt.show()
    
    print(f"Predicted Sign : {CLASS_NAMES[top_class]}")
    print(f"Class ID       : {top_class}")
    print(f"Confidence     : {confidence:.2f}%")

# Test on a sample Yield sign (Class 13)
sample_folder = os.path.join(DATA_PATH, "13")
if os.path.exists(sample_folder) and len(os.listdir(sample_folder)) > 0:
    sample_file = os.path.join(sample_folder, os.listdir(sample_folder)[0])
    print("Testing on a Yield sign (Class 13):")
    predict_traffic_sign(sample_file)

print("✅ Phase 9 Complete: Inference function working.")

## 📌 Phase 10: Model Export for Web Deployment
We save the trained weights to `model/traffic_sign_model.keras` so the Flask web interface (`app.py`) can load it directly.

In [ ]:
os.makedirs("model", exist_ok=True)
export_path = os.path.join("model", "traffic_sign_model.keras")

model.save(export_path)
print(f"🎉 SUCCESS: Model exported to '{export_path}'")
print("✅ Phase 10 Complete: You can now run 'python app.py' to launch the web dashboard!")